# B2 - Baseline forecasters

The Lago et al. (2021) EPF benchmark suite:

* **NaiveForecaster** - `ŷ_{t+h} = y_t`. Anchored on `price_lag1h`.
* **SeasonalNaiveForecaster(24h)** - `ŷ_{t+h} = price[t+h-24]`.
* **SeasonalNaiveForecaster(168h)** - `ŷ_{t+h} = price[t+h-168]` (yesterday-of-week).

All three produce quantile bands via empirical residual bootstrap. Any
model in B3/B4 must beat the seasonal-168h baseline with statistical
significance to be worth shipping.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'src').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)

## Load splits and build features

In [ ]:
from data.loaders import load_split
from module_b.features import prepare_supervised, HORIZON_COL, TARGET_COL
from module_b.features import build_features

train_df = load_split('train')
val_df = load_split('val')
full = pd.concat([train_df, val_df])

feat = build_features(full, bundles=('calendar', 'lags'))
past_cols = ['price_lag1h', 'price_lag24h', 'price_lag168h', 'price_rmean24h']
future_cols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
               'is_weekend', 'is_holiday_DE']
X, y = prepare_supervised(feat, horizons=range(1, 25),
                          past_cols=past_cols, future_cols=future_cols)

train_mask = X['origin_ts'] < val_df.index.min()
X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[~train_mask], y.loc[~train_mask]
print(f'train rows: {len(X_train):,}, val rows: {len(X_val):,}')

## Fit all three baselines

In [ ]:
from module_b.models import NaiveForecaster, SeasonalNaiveForecaster

price_series = full['price']

naive = NaiveForecaster().fit(X_train, y_train)
sn24 = SeasonalNaiveForecaster(season_hours=24).fit(
    X_train, y_train, price_series=price_series)
sn168 = SeasonalNaiveForecaster(season_hours=168).fit(
    X_train, y_train, price_series=price_series)

models = {'naive': naive, 'seasonal_24h': sn24,
          'seasonal_168h': sn168}
print('Fitted:', list(models))

## Predict on val and compute metrics

In [ ]:
from module_b.features import HorizonGroup, HORIZON_RANGES
from module_b.evaluation import mae, pinball_loss, multi_pinball_loss, coverage

rows = []
for name, m in models.items():
    q = m.predict_quantiles(X_val)
    yv = y_val.to_numpy()
    for group in HorizonGroup:
        horizons = HORIZON_RANGES[group]
        mask = X_val[HORIZON_COL].isin(horizons).to_numpy()
        if not mask.any():
            continue
        rows.append({
            'model': name,
            'horizon_group': group.value,
            'mae': mae(yv[mask], q['q50'].to_numpy()[mask]),
            'pinball_avg': multi_pinball_loss(yv[mask], q.iloc[mask], [0.1, 0.5, 0.9]),
            'coverage_80': coverage(yv[mask], q['q10'].to_numpy()[mask], q['q90'].to_numpy()[mask]),
        })

results = pd.DataFrame(rows)
results.set_index(['horizon_group', 'model']).round(3)

## Pivot to compare models per horizon group

In [ ]:
pivot = results.pivot_table(index='model', columns='horizon_group',
                            values=['mae', 'pinball_avg', 'coverage_80']).round(3)
pivot

## Visualise: per-horizon pinball loss

In [ ]:
from module_b.evaluation import multi_pinball_loss

fig, ax = plt.subplots(figsize=(11, 4))
for name, m in models.items():
    q = m.predict_quantiles(X_val)
    per_h = []
    for h in range(1, 25):
        mask = (X_val[HORIZON_COL] == h).to_numpy()
        if mask.any():
            per_h.append(multi_pinball_loss(
                y_val.to_numpy()[mask], q.iloc[mask], [0.1, 0.5, 0.9]))
        else:
            per_h.append(np.nan)
    ax.plot(range(1, 25), per_h, marker='.', label=name)
ax.set_xlabel('Horizon (h)')
ax.set_ylabel('Avg pinball loss (lower = better)')
ax.set_title('Baseline comparison on 2024 val split')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Save results to disk for use in later notebooks

In [ ]:
outputs_dir = REPO_ROOT / 'notebooks' / 'module_b' / 'outputs'
outputs_dir.mkdir(exist_ok=True)
results.to_parquet(outputs_dir / 'B2_baseline_results.parquet')
print(f'Wrote {outputs_dir / "B2_baseline_results.parquet"}')